# YUNESA Academic GraphRAG with Groq

Notebook ini adalah development notebook untuk menguji GraphRAG end-to-end sesuai konsep tugas akhir:

1. Metadata akademik dari Supabase.
2. Knowledge graph ontology di Neo4j/AuraDB.
3. Dual vector index di Zilliz/Milvus: `PaperChunk`, `EntityEmbedding`, `RelationshipEmbedding`, `ContentKeyword`.
4. Retrieval mode ala AcademicRAG: `naive`, `subgraph`, `global`, `hybrid`, dan `mix`.
5. Answer synthesis memakai Groq, dengan prompt ketat berbasis evidence.

Notebook ini tidak mencetak secret. Semua credential dibaca dari `.env` atau environment runtime.

In [ ]:
from pathlib import Path
import importlib.util
import json
import os
import subprocess
import sys

def is_colab_runtime() -> bool:
    return (
        'google.colab' in sys.modules
        or bool(os.getenv('COLAB_RELEASE_TAG'))
        or importlib.util.find_spec('google.colab') is not None
    )

IN_COLAB = is_colab_runtime()
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

PROJECT_ROOT = Path(os.getenv('YUNESA_PROJECT_DIR', '/content/drive/MyDrive/Tugas_Akhir' if IN_COLAB else '../..')).resolve()
BUILD_GRAPH_DIR = PROJECT_ROOT / 'notebooks' / 'build-graph'
SRC_DIR = BUILD_GRAPH_DIR / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

required_modules = ['pandas', 'networkx', 'rdflib', 'supabase', 'neo4j', 'pymilvus', 'requests', 'groq', 'dotenv', 'opik']
missing = [module for module in required_modules if importlib.util.find_spec(module) is None]
if missing and IN_COLAB:
    pip_packages = [
        'pandas', 'networkx', 'rdflib==6.3.1', 'supabase>=2.25.1', 'python-dotenv', 'opik>=1.0.0',
        'neo4j>=6.1.0', 'pymilvus>=2.6.5', 'requests', 'groq>=1.1.2'
    ]
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *pip_packages])
elif missing:
    raise RuntimeError('Missing notebook dependencies: ' + ', '.join(missing))

from yunesa_academic_kg import (
    GraphRAGGenerationParam,
    GraphRAGQueryParam,
    KGConfig,
    fetch_supabase_sample,
    format_graphrag_context,
    generate_graphrag_answer_with_groq,
    graphrag_answer,
    graphrag_retrieve,
    graph_quality_report,
    inspect_milvus_collections,
    inspect_neo4j_graph,
    load_local_csv_sample,
    load_project_env,
    milvus_credential_status,
    neo4j_credential_status,
    run_local_kg_pipeline,
    supabase_credential_status,
)

os.environ.setdefault('NEO4J_TRUST_SELF_SIGNED', '1')
os.environ.setdefault('HF_HUB_DISABLE_PROGRESS_BARS', '1')
os.environ.setdefault('TOKENIZERS_PARALLELISM', 'false')
os.environ.setdefault('TRANSFORMERS_VERBOSITY', 'error')

config = KGConfig.default(sample_size=int(os.getenv('YUNESA_SAMPLE_SIZE', '5')))
load_project_env(config.project_root)
GRAPH_NAME = os.getenv('YUNESA_GRAPH_NAME', 'yunesa_academic_kg_debug_20260604')
KEYWORD_PROVIDER = os.getenv('YUNESA_GRAPHRAG_KEYWORD_PROVIDER', 'heuristic')
KEYWORD_CACHE = os.getenv(
    'YUNESA_GRAPHRAG_KEYWORD_CACHE',
    str(BUILD_GRAPH_DIR / 'outputs' / 'academic_kg' / 'graphrag_keyword_cache.json'),
)

print('Project root:', config.project_root)
print('Graph namespace:', GRAPH_NAME)
print('Supabase credentials:', supabase_credential_status())
print('Neo4j credentials:', neo4j_credential_status())
print('Milvus credentials:', milvus_credential_status())
print('Groq configured:', bool(os.getenv('GROQ_API_KEY')))
print('Opik enabled:', os.getenv('OPIK_ENABLED', 'true').lower() not in {'0', 'false', 'no', 'off'} and bool(os.getenv('OPIK_API_KEY') or os.getenv('OPIK_URL_OVERRIDE') or os.getenv('OPIK_USE_LOCAL')))

## 1. Supabase Data Sanity Check

Cell ini mengecek karakteristik data paper yang menjadi sumber KG. Jika Supabase tidak tersedia, notebook fallback ke CSV lokal.

In [ ]:
try:
    papers_df, lecturers_df, links_df = fetch_supabase_sample(sample_size=config.sample_size)
    data_source = 'supabase'
except Exception as exc:
    print('Supabase unavailable, fallback to local CSV:', type(exc).__name__, exc)
    papers_df, lecturers_df, links_df = load_local_csv_sample(config.project_root / 'notebooks', sample_size=config.sample_size)
    data_source = 'local_csv'

required_paper_cols = ['title', 'abstract', 'tldr', 'keywords', 'authors']
missing_summary = {col: int(papers_df.get(col, '').fillna('').astype(str).str.strip().eq('').sum()) for col in required_paper_cols if col in papers_df.columns}
print('Data source:', data_source)
print('Papers:', len(papers_df), 'Lecturers:', len(lecturers_df), 'Paper-lecturer links:', len(links_df))
print('Missing summary:', missing_summary)
papers_df[['title', 'year', 'journal', 'document_type', 'tldr']].head(5)

## 2. Optional KG Construction Refresh

Default-nya tidak menulis ulang database. Aktifkan hanya saat perlu rebuild sample graph:

```python
os.environ['YUNESA_RUN_CONSTRUCTION'] = '1'
os.environ['YUNESA_WRITE_NEO4J'] = '1'
os.environ['YUNESA_WRITE_MILVUS'] = '1'
os.environ['YUNESA_CLEAR_GRAPH'] = '1'
os.environ['YUNESA_USE_EXTRACTION'] = '1'
```

`YUNESA_CLEAR_GRAPH=1` membersihkan graph/row untuk `GRAPH_NAME`, bukan seluruh project.

In [ ]:
RUN_CONSTRUCTION = os.getenv('YUNESA_RUN_CONSTRUCTION', '0') == '1'
if RUN_CONSTRUCTION:
    result = run_local_kg_pipeline(
        sample_size=config.sample_size,
        source='supabase',
        graph_name=GRAPH_NAME,
        write_neo4j=os.getenv('YUNESA_WRITE_NEO4J', '0') == '1',
        write_milvus=os.getenv('YUNESA_WRITE_MILVUS', '0') == '1',
        clear_neo4j=os.getenv('YUNESA_CLEAR_GRAPH', '0') == '1',
        clear_milvus=os.getenv('YUNESA_CLEAR_GRAPH', '0') == '1',
        use_extraction=os.getenv('YUNESA_USE_EXTRACTION', '1') == '1',
    )
    print(json.dumps({
        'validation': result['validation'],
        'quality_gates': result['quality']['quality_gates'],
        'storage_reports': result['storage_reports'],
    }, indent=2, ensure_ascii=False, default=str))
else:
    print('KG construction refresh skipped. Using existing AuraDB/Zilliz graph namespace:', GRAPH_NAME)

## 3. Storage Layer Inspection

AcademicRAG-style storage layer pada project ini:

- Supabase: metadata akademik (`papers`, `lecturers`, `paper_lecturers`).
- AuraDB/Neo4j: ontology graph dan traversal relasi akademik.
- Zilliz/Milvus: vector retrieval untuk chunk, entity, relationship, dan keyword.

In [ ]:
try:
    neo_report = inspect_neo4j_graph(graph_name=GRAPH_NAME)
    print('Neo4j nodes:', neo_report['nodes'], 'relationships:', neo_report['relationships'])
    print('Neo4j labels:', neo_report['label_counts'])
    print('Neo4j relations:', neo_report['relationship_counts'])
except Exception as exc:
    print('Neo4j inspect failed:', type(exc).__name__, exc)

try:
    milvus_report = inspect_milvus_collections()
    compact_milvus = {
        name: {
            'row_count_stat': (data.get('stats') or {}).get('row_count'),
            'fields': [field.get('name') for field in data.get('fields', [])],
        }
        for name, data in (milvus_report.get('collections') or {}).items()
    }
    print(json.dumps(compact_milvus, indent=2, ensure_ascii=False))
except Exception as exc:
    print('Milvus inspect failed:', type(exc).__name__, exc)

## 4. Retrieval Mode Diagnostics

Mode yang diuji mengikuti ide AcademicRAG:

- `naive`: vector search pada `PaperChunk`.
- `subgraph`: entity vector search + Neo4j neighborhood.
- `global`: relationship vector search.
- `hybrid`: entity + relationship.
- `mix`: chunk + keyword + entity + relationship + subgraph.

In [ ]:
QUERY = os.getenv('YUNESA_GRAPHRAG_QUERY', 'Model apa yang digunakan untuk race and gender recognition dan dataset apa yang dipakai?')
base_query_param = dict(
    top_k=3,
    graph_name=GRAPH_NAME,
    keyword_provider=KEYWORD_PROVIDER,
    keyword_cache_path=KEYWORD_CACHE,
    keyword_top_k=8,
    max_keyword_terms=8,
)

for mode in ['naive', 'subgraph', 'global', 'hybrid', 'mix']:
    print('\n' + '=' * 100)
    print('MODE:', mode)
    retrieval = graphrag_retrieve(
        QUERY,
        param=GraphRAGQueryParam(mode=mode, **base_query_param),
    )
    print(format_graphrag_context(retrieval, max_chars=4500))


## 5. Groq Answer Synthesis

Cell ini memanggil Groq. Prompt dibatasi agar jawaban hanya berdasarkan evidence hasil retrieval dan graph traversal.

In [ ]:
retrieval = graphrag_retrieve(
    QUERY,
    param=GraphRAGQueryParam(mode='mix', **base_query_param),
)
print('Keyword decomposition:')
print(json.dumps(retrieval.get('keyword_decomposition', {}), indent=2, ensure_ascii=False))

answer = generate_graphrag_answer_with_groq(
    QUERY,
    retrieval,
    param=GraphRAGGenerationParam(
        model=os.getenv('GROQ_GRAPHRAG_MODEL', 'llama-3.3-70b-versatile'),
        temperature=0.1,
        max_tokens=700,
        context_max_chars=12000,
    ),
)
print('ANSWER:\n', answer['answer'])
print('\nUSAGE:', answer['usage'])
print('\nSOURCES:')
print(json.dumps(answer['sources'], indent=2, ensure_ascii=False))


## 6. Minimal Evaluation Harness

Evaluasi ringan ini mengecek apakah jawaban mengandung entitas yang seharusnya muncul. Ini bukan evaluasi final, tetapi cukup untuk regression test awal saat prompt/retrieval berubah.

In [ ]:
test_cases = [
    {
        'query': 'Model apa yang digunakan untuk race and gender recognition dan dataset apa yang dipakai?',
        'expected_terms': ['ViT-Face', 'ViT-Emotion', 'DemogPairs'],
    },
    {
        'query': 'Paper mana yang membahas credit default risk prediction dan model boosting apa yang digunakan?',
        'expected_terms': ['XGBoost', 'CatBoost', 'LightGBM', 'Optuna'],
    },
]

eval_rows = []
for case in test_cases:
    retrieval = graphrag_retrieve(case['query'], param=GraphRAGQueryParam(mode='mix', **base_query_param))
    generated = generate_graphrag_answer_with_groq(
        case['query'],
        retrieval,
        param=GraphRAGGenerationParam(max_tokens=500),
    )
    answer_text = generated['answer']
    found_terms = [term for term in case['expected_terms'] if term.lower() in answer_text.lower()]
    eval_rows.append({
        'query': case['query'],
        'keyword_decomposition': retrieval.get('keyword_decomposition', {}),
        'expected_terms': case['expected_terms'],
        'found_terms': found_terms,
        'pass': len(found_terms) == len(case['expected_terms']),
        'usage': generated['usage'],
        'answer': answer_text,
    })

print(json.dumps(eval_rows, indent=2, ensure_ascii=False))
